In [14]:
# Sample training data: list of sentences with (word, tag)
train_data = [
    [('The', 'DT'), ('cat', 'NN'), ('sat', 'VBD')],
    [('A', 'DT'), ('dog', 'NN'), ('barked', 'VBD')],
    [('The', 'DT'), ('dog', 'NN'), ('ran', 'VBD')],
    [('Cats', 'NNS'), ('are', 'VBP'), ('cute', 'JJ')]
]


In [15]:
from collections import defaultdict, Counter

# Count occurrences
emission_counts = defaultdict(Counter)
transition_counts = defaultdict(Counter)
tag_counts = Counter()

# Special start and end symbols
START = '<START>'
END = '<END>'

for sentence in train_data:
    prev_tag = START
    for word, tag in sentence:
        emission_counts[tag][word] += 1
        transition_counts[prev_tag][tag] += 1
        tag_counts[tag] += 1
        prev_tag = tag
    transition_counts[prev_tag][END] += 1

# Convert counts to probabilities
states = list(tag_counts.keys())

# Emission probabilities: P(word|tag)
emission_prob = {}
for tag in states:
    total = sum(emission_counts[tag].values())
    emission_prob[tag] = {word: count/total for word, count in emission_counts[tag].items()}

# Transition probabilities: P(curr_tag|prev_tag)
transition_prob = {}
for prev_tag, counter in transition_counts.items():
    total = sum(counter.values())
    transition_prob[prev_tag] = {tag: count/total for tag, count in counter.items()}


In [16]:
def viterbi(obs, states, start_symbol, transition_prob, emission_prob):
    V = [{}]  # Probability table
    backpointer = [{}]

    # Initialization
    for s in states:
        V[0][s] = transition_prob.get(start_symbol, {}).get(s, 0) * emission_prob.get(s, {}).get(obs[0], 0)
        backpointer[0][s] = start_symbol

    # Recursion
    for t in range(1, len(obs)):
        V.append({})
        backpointer.append({})
        for s in states:
            max_prob, prev_state = max(
                [(V[t-1][s0] * transition_prob.get(s0, {}).get(s, 0) * emission_prob.get(s, {}).get(obs[t], 0), s0)
                 for s0 in states],
                key=lambda x: x[0]
            )
            V[t][s] = max_prob
            backpointer[t][s] = prev_state

    # Termination
    max_prob, last_tag = max([(V[-1][s] * transition_prob.get(s, {}).get(END, 0), s) for s in states], key=lambda x: x[0])

    # Backtrack
    best_path = [last_tag]
    for t in range(len(obs)-1, 0, -1):
        best_path.insert(0, backpointer[t][best_path[0]])

    return best_path


In [17]:
test_sentence = ['The', 'dog', 'sat']
tags = viterbi(test_sentence, states, START, transition_prob, emission_prob)
print(list(zip(test_sentence, tags)))


[('The', 'DT'), ('dog', 'NN'), ('sat', 'VBD')]


In [18]:
import nltk
from nltk.corpus import treebank

nltk.download('treebank')
train_sents = treebank.tagged_sents()[:3000]
tagger = nltk.UnigramTagger(train_sents)

print(tagger.tag(['The', 'dog', 'sat']))


[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.


[('The', 'DT'), ('dog', None), ('sat', None)]


In [19]:
import math

# Vocabulary and states
vocab = set(word for tag_dict in emission_counts.values() for word in tag_dict)
states = list(tag_counts.keys())

# Add-one Laplace smoothing
def smoothed_emission(tag, word, alpha=1e-3):
    total = sum(emission_counts[tag].values()) + alpha * (len(vocab) + 1)  # +1 for unknown words
    count = emission_counts[tag].get(word, 0) + alpha
    return count / total

def smoothed_transition(prev_tag, curr_tag, alpha=1e-3):
    total = sum(transition_counts[prev_tag].values()) + alpha * (len(states) + 1)
    count = transition_counts[prev_tag].get(curr_tag, 0) + alpha
    return count / total


In [20]:
def viterbi_log(obs, states, start_symbol):
    V = [{}]
    backpointer = [{}]

    # Initialization
    for s in states:
        prob = math.log(smoothed_transition(start_symbol, s)) + math.log(smoothed_emission(s, obs[0]))
        V[0][s] = prob
        backpointer[0][s] = start_symbol

    # Recursion
    for t in range(1, len(obs)):
        V.append({})
        backpointer.append({})
        for s in states:
            max_prob, prev_state = max(
                [(V[t-1][s0] + math.log(smoothed_transition(s0, s)) + math.log(smoothed_emission(s, obs[t])), s0)
                 for s0 in states],
                key=lambda x: x[0]
            )
            V[t][s] = max_prob
            backpointer[t][s] = prev_state

    # Termination
    max_prob, last_tag = max(
        [(V[-1][s] + math.log(smoothed_transition(s, END)), s) for s in states],
        key=lambda x: x[0]
    )

    # Backtrack
    best_path = [last_tag]
    for t in range(len(obs)-1, 0, -1):
        best_path.insert(0, backpointer[t][best_path[0]])

    return best_path


In [21]:
test_sentence = ['The', 'dog', 'ran', 'fast']  # 'fast' might be unseen
tags = viterbi_log(test_sentence, states, START)
print(list(zip(test_sentence, tags)))


[('The', 'DT'), ('dog', 'NN'), ('ran', 'VBD'), ('fast', 'JJ')]


In [22]:
# =======================================
# HMM-based POS Tagger from Scratch
# =======================================

import math
from collections import defaultdict, Counter

# =========================
# Step 0: Training Data
# Replace this with IIT Virtual Lab corpus
# Format: list of sentences, each sentence = list of (word, tag) tuples
train_data = [
    [('The', 'DT'), ('cat', 'NN'), ('sat', 'VBD')],
    [('A', 'DT'), ('dog', 'NN'), ('barked', 'VBD')],
    [('The', 'DT'), ('dog', 'NN'), ('ran', 'VBD')],
    [('Cats', 'NNS'), ('are', 'VBP'), ('cute', 'JJ')]
]

START = '<START>'
END = '<END>'

# =========================
# Step 1: Compute Counts
# =========================

emission_counts = defaultdict(Counter)
transition_counts = defaultdict(Counter)
tag_counts = Counter()

for sentence in train_data:
    prev_tag = START
    for word, tag in sentence:
        emission_counts[tag][word] += 1
        transition_counts[prev_tag][tag] += 1
        tag_counts[tag] += 1
        prev_tag = tag
    transition_counts[prev_tag][END] += 1

states = list(tag_counts.keys())
vocab = set(word for tag_dict in emission_counts.values() for word in tag_dict)

# =========================
# Step 2: Smoothed Probabilities
# =========================

alpha = 1e-3  # smoothing factor

def smoothed_emission(tag, word):
    total = sum(emission_counts[tag].values()) + alpha * (len(vocab) + 1)
    count = emission_counts[tag].get(word, 0) + alpha
    return count / total

def smoothed_transition(prev_tag, curr_tag):
    total = sum(transition_counts[prev_tag].values()) + alpha * (len(states) + 1)
    count = transition_counts[prev_tag].get(curr_tag, 0) + alpha
    return count / total

# =========================
# Step 3: Viterbi with Log Probabilities
# =========================

def viterbi_log(obs, states, start_symbol=START):
    V = [{}]
    backpointer = [{}]

    # Initialization
    for s in states:
        V[0][s] = math.log(smoothed_transition(start_symbol, s)) + math.log(smoothed_emission(s, obs[0]))
        backpointer[0][s] = start_symbol

    # Recursion
    for t in range(1, len(obs)):
        V.append({})
        backpointer.append({})
        for s in states:
            max_prob, prev_state = max(
                [(V[t-1][s0] + math.log(smoothed_transition(s0, s)) + math.log(smoothed_emission(s, obs[t])), s0)
                 for s0 in states],
                key=lambda x: x[0]
            )
            V[t][s] = max_prob
            backpointer[t][s] = prev_state

    # Termination
    max_prob, last_tag = max(
        [(V[-1][s] + math.log(smoothed_transition(s, END)), s) for s in states],
        key=lambda x: x[0]
    )

    # Backtrack
    best_path = [last_tag]
    for t in range(len(obs)-1, 0, -1):
        best_path.insert(0, backpointer[t][best_path[0]])

    return best_path

# =========================
# Step 4: Test the HMM Tagger
# =========================

test_sentence = ['The', 'dog', 'ran', 'fast']  # 'fast' might be unseen
tags = viterbi_log(test_sentence, states)
print("HMM POS tagging (from scratch):")
print(list(zip(test_sentence, tags)))

# =========================
# Step 5: Ready-made library comparison (NLTK)
# =========================

import nltk
from nltk.corpus import treebank

nltk.download('treebank')
train_sents = treebank.tagged_sents()[:3000]

tagger = nltk.UnigramTagger(train_sents)
print("\nNLTK UnigramTagger POS tagging:")
print(tagger.tag(['The', 'dog', 'ran', 'fast']))


HMM POS tagging (from scratch):
[('The', 'DT'), ('dog', 'NN'), ('ran', 'VBD'), ('fast', 'JJ')]


[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!



NLTK UnigramTagger POS tagging:
[('The', 'DT'), ('dog', None), ('ran', 'VBD'), ('fast', 'RB')]
